## Gold Layer — Business Analytics (Incremental MERGE)

This notebook builds **6 business-ready gold tables** from the `finance.silver` layer:

| # | Gold Table | Business Question | Source Tables |
|---|-----------|-------------------|---------------|
| 1 | `stock_profitability` | Which stocks generate the highest profit? | fact_transactions, dim_stock |
| 2 | `portfolio_performance` | How does portfolio value change over time? | fact_portfolio_snapshot, fact_prices, dim_date |
| 3 | `trading_behavior` | What is the BUY vs SELL ratio by sector? | fact_transactions, dim_stock |
| 4 | `market_event_impact` | How do events affect prices & volume? | dim_event, fact_prices, fact_transactions |
| 5 | `sector_performance` | Which sectors deliver highest returns? | dim_stock, fact_prices, dim_date |
| 6 | `user_investment_patterns` | Who are high-freq traders vs long-term? | fact_transactions, dim_date |

In [0]:
# =========================
# GOLD LAYER — Business Analytics (Incremental MERGE)
# =========================
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, sum as _sum, count, avg, min as _min, max as _max,
    datediff, when, lit, date_format, round as _round,
    countDistinct, lag, row_number, dense_rank
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

SILVER = "finance.silver"
GOLD = "finance.gold"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

def upsert(df_source, table_name, merge_key):
    """Incremental upsert: INSERT new, UPDATE existing."""
    full_table = f"{GOLD}.{table_name}"
    df_source = df_source.withColumn("_loaded_at", F.current_timestamp())
    if spark.catalog.tableExists(full_table):
        dt = DeltaTable.forName(spark, full_table)
        dt.alias("t").merge(
            df_source.alias("s"), f"t.{merge_key} = s.{merge_key}"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
        action = "MERGED"
    else:
        df_source.write.format("delta").saveAsTable(full_table)
        action = "CREATED"
    cnt = spark.table(full_table).count()
    print(f"   {full_table} — {cnt:,} rows ({action})")

print(" Gold analytics config ready")

In [0]:
# =========================
# 1. STOCK PROFITABILITY ANALYSIS
# Which stocks generate the highest profit for users?
# Logic: SELL revenue − BUY cost per user per stock
# =========================

df_txn = spark.table(f"{SILVER}.fact_transactions").filter(col("status") == "COMPLETED")
df_stock = spark.table(f"{SILVER}.dim_stock")

# Aggregate BUY and SELL per user per stock
df_buys = (
    df_txn.filter(col("txn_type") == "BUY")
    .groupBy("user_id", "stock_id")
    .agg(
        _sum("total_value").alias("total_buy_value"),
        _sum("quantity").alias("total_buy_qty")
    )
)

df_sells = (
    df_txn.filter(col("txn_type") == "SELL")
    .groupBy("user_id", "stock_id")
    .agg(
        _sum("total_value").alias("total_sell_value"),
        _sum("quantity").alias("total_sell_qty")
    )
)

df_profit = (
    df_buys.join(df_sells, ["user_id", "stock_id"], "full")
    .join(df_stock, "stock_id", "left")
    .fillna(0)
    .withColumn("profit", _round(col("total_sell_value") - col("total_buy_value"), 2))
    .withColumn("roi_pct", _round(
        when(col("total_buy_value") > 0,
             (col("total_sell_value") - col("total_buy_value")) / col("total_buy_value") * 100
        ).otherwise(0), 2))
    .withColumn("profit_key", F.concat_ws("_", "user_id", "stock_id"))
    .select("profit_key", "user_id", "stock_id", "ticker", "company_name", "sector",
            "total_buy_value", "total_buy_qty", "total_sell_value", "total_sell_qty",
            "profit", "roi_pct")
)

print(" STOCK PROFITABILITY")
upsert(df_profit, "stock_profitability", "profit_key")

In [0]:
# =========================
# 2. PORTFOLIO PERFORMANCE OVER TIME
# How does a user's portfolio value change daily/monthly?
# Logic: snapshot quantity × closing price = market value per day
# =========================

df_snap = spark.table(f"{SILVER}.fact_portfolio_snapshot")
df_prices = spark.table(f"{SILVER}.fact_prices")
df_date = spark.table(f"{SILVER}.dim_date")
df_stock = spark.table(f"{SILVER}.dim_stock")

# Join snapshot with closing prices on (stock_id, date_id)
df_daily = (
    df_snap
    .join(df_prices, (df_snap.stock_id == df_prices.stock_id) & 
                      (df_snap.snapshot_date_id == df_prices.date_id), "inner")
    .join(df_date, df_snap.snapshot_date_id == df_date.date_id, "left")
    .join(df_stock, df_snap.stock_id == df_stock.stock_id, "left")
    .select(
        df_snap.portfolio_id, df_snap.user_id, df_snap.stock_id,
        df_stock.ticker, df_stock.sector,
        df_date.full_date.alias("snapshot_date"),
        df_date.year, df_date.month, df_date.month_name,
        df_snap.quantity,
        df_snap.avg_cost,
        df_snap.total_invested,
        df_prices.close.alias("close_price"),
        _round(df_snap.quantity * df_prices.close, 2).alias("market_value"),
        _round(df_snap.quantity * df_prices.close - df_snap.total_invested, 2).alias("unrealized_pnl"),
        _round(
            when(df_snap.total_invested > 0,
                 (df_snap.quantity * df_prices.close - df_snap.total_invested) / df_snap.total_invested * 100
            ).otherwise(0), 2
        ).alias("unrealized_pnl_pct")
    )
)

# Daily aggregation per user
df_portfolio_daily = (
    df_daily
    .groupBy("user_id", "snapshot_date", "year", "month", "month_name")
    .agg(
        _round(_sum("total_invested"), 2).alias("total_invested"),
        _round(_sum("market_value"), 2).alias("total_market_value"),
        _round(_sum("unrealized_pnl"), 2).alias("total_unrealized_pnl"),
        countDistinct("stock_id").alias("num_holdings")
    )
    .withColumn("pnl_pct", _round(
        when(col("total_invested") > 0,
             col("total_unrealized_pnl") / col("total_invested") * 100
        ).otherwise(0), 2))
    .withColumn("perf_key", F.concat_ws("_", "user_id", "snapshot_date"))
)

print(" PORTFOLIO PERFORMANCE OVER TIME")
upsert(df_portfolio_daily, "portfolio_performance", "perf_key")

In [0]:
# =========================
# 3. TRADING BEHAVIOR ANALYSIS
# BUY vs SELL ratio across users and sectors
# =========================

df_txn = spark.table(f"{SILVER}.fact_transactions").filter(col("status") == "COMPLETED")
df_stock = spark.table(f"{SILVER}.dim_stock")
df_date = spark.table(f"{SILVER}.dim_date")

df_behavior = (
    df_txn
    .join(df_stock, "stock_id", "left")
    .join(df_date, "date_id", "left")
    .groupBy("user_id", "sector", "year", "month")
    .agg(
        count(when(col("txn_type") == "BUY", True)).alias("buy_count"),
        count(when(col("txn_type") == "SELL", True)).alias("sell_count"),
        _sum(when(col("txn_type") == "BUY", col("total_value")).otherwise(0)).alias("buy_volume"),
        _sum(when(col("txn_type") == "SELL", col("total_value")).otherwise(0)).alias("sell_volume"),
        _sum("fee").alias("total_fees"),
        count("*").alias("total_txns")
    )
    .withColumn("buy_sell_ratio", _round(
        when(col("sell_count") > 0, col("buy_count") / col("sell_count"))
        .otherwise(lit(None)), 2))
    .withColumn("net_flow", _round(col("buy_volume") - col("sell_volume"), 2))
    .withColumn("behavior_key", F.concat_ws("_", "user_id", "sector", "year", "month"))
)

print(" TRADING BEHAVIOR ANALYSIS")
upsert(df_behavior, "trading_behavior", "behavior_key")

In [0]:
# =========================
# 4. IMPACT OF MARKET EVENTS
# How do market events affect stock prices and trading volume?
# Logic: Compare price & volume on event day vs prior day
# =========================

df_event = spark.table(f"{SILVER}.dim_event")
df_prices = spark.table(f"{SILVER}.fact_prices")
df_txn = spark.table(f"{SILVER}.fact_transactions").filter(col("status") == "COMPLETED")
df_stock = spark.table(f"{SILVER}.dim_stock")
df_date = spark.table(f"{SILVER}.dim_date")

# Get prior day price using window function
w = Window.partitionBy("stock_id").orderBy("date_id")
df_prices_lag = df_prices.withColumn("prev_close", lag("close").over(w))

# Trading volume on event day per stock
df_txn_vol = (
    df_txn
    .groupBy("stock_id", "date_id")
    .agg(
        count("*").alias("txn_count"),
        _sum("quantity").alias("traded_qty"),
        _sum("total_value").alias("traded_value")
    )
)

# Join everything on event
df_event_impact = (
    df_event
    .join(df_prices_lag, ["stock_id", "date_id"], "left")
    .join(df_txn_vol, ["stock_id", "date_id"], "left")
    .join(df_stock, "stock_id", "left")
    .join(df_date, "date_id", "left")
    .select(
        col("event_id"),
        df_event.stock_id,
        col("ticker"), col("company_name"), col("sector"),
        col("full_date").alias("event_date"),
        col("event_type"), col("impact"), col("description"),
        col("price_change_pct"),
        _round(col("prev_close"), 2).alias("prev_close"),
        _round(col("close"), 2).alias("event_close"),
        _round(col("close") - col("prev_close"), 2).alias("price_change_abs"),
        col("volume").alias("market_volume"),
        col("txn_count").alias("user_txn_count"),
        col("traded_qty").alias("user_traded_qty"),
        _round(col("traded_value"), 2).alias("user_traded_value")
    )
    .fillna(0, subset=["user_txn_count", "user_traded_qty", "user_traded_value"])
)

print("4\ufe0f\u20e3 MARKET EVENTS IMPACT")
upsert(df_event_impact, "market_event_impact", "event_id")

In [0]:
# =========================
# 5. TOP PERFORMING SECTORS
# Which sectors deliver the highest returns over time?
# Logic: Sector-level monthly return = (close_end − close_start) / close_start
# =========================

df_prices = spark.table(f"{SILVER}.fact_prices")
df_stock = spark.table(f"{SILVER}.dim_stock")
df_date = spark.table(f"{SILVER}.dim_date")
df_txn = spark.table(f"{SILVER}.fact_transactions").filter(col("status") == "COMPLETED")

# Monthly first and last close per stock
w_asc = Window.partitionBy("stock_id", "year", "month").orderBy("date_id")
w_desc = Window.partitionBy("stock_id", "year", "month").orderBy(F.desc("date_id"))

df_monthly_prices = (
    df_prices
    .join(df_date, "date_id", "left")
    .withColumn("rn_first", row_number().over(w_asc))
    .withColumn("rn_last", row_number().over(w_desc))
)

df_open_month = df_monthly_prices.filter(col("rn_first") == 1).select(
    "stock_id", "year", "month", col("close").alias("month_open_close")
)
df_close_month = df_monthly_prices.filter(col("rn_last") == 1).select(
    "stock_id", "year", "month", col("close").alias("month_end_close")
)

# Transaction volume per sector per month
df_sector_txn = (
    df_txn.join(df_stock, "stock_id", "left")
    .join(df_date, "date_id", "left")
    .groupBy("sector", "year", "month")
    .agg(
        _sum("total_value").alias("total_traded_value"),
        _sum("quantity").alias("total_traded_qty"),
        count("*").alias("total_txns")
    )
)

# Sector-level monthly returns
df_sector_perf = (
    df_open_month
    .join(df_close_month, ["stock_id", "year", "month"], "inner")
    .join(df_stock.select("stock_id", "sector", "ticker"), "stock_id", "left")
    .groupBy("sector", "year", "month")
    .agg(
        _round(avg("month_open_close"), 2).alias("avg_month_open"),
        _round(avg("month_end_close"), 2).alias("avg_month_close"),
        countDistinct("stock_id").alias("num_stocks")
    )
    .withColumn("monthly_return_pct", _round(
        when(col("avg_month_open") > 0,
             (col("avg_month_close") - col("avg_month_open")) / col("avg_month_open") * 100
        ).otherwise(0), 2))
    .join(df_sector_txn, ["sector", "year", "month"], "left")
    .fillna(0)
    .withColumn("sector_key", F.concat_ws("_", "sector", "year", "month"))
)

print(" TOP PERFORMING SECTORS")
upsert(df_sector_perf, "sector_performance", "sector_key")

In [0]:
# =========================
# 6. USER INVESTMENT PATTERNS
# Which users are high-frequency traders vs long-term investors?
# Logic: Count transactions, avg holding duration, trading frequency
# =========================

df_txn = spark.table(f"{SILVER}.fact_transactions").filter(col("status") == "COMPLETED")
df_date = spark.table(f"{SILVER}.dim_date")

# Get actual dates for transactions
df_txn_dated = df_txn.join(df_date, "date_id", "left")

# Per-user overall stats
df_user_stats = (
    df_txn_dated
    .groupBy("user_id")
    .agg(
        count("*").alias("total_txns"),
        count(when(col("txn_type") == "BUY", True)).alias("buy_txns"),
        count(when(col("txn_type") == "SELL", True)).alias("sell_txns"),
        _round(_sum("total_value"), 2).alias("total_value_traded"),
        _round(avg("total_value"), 2).alias("avg_txn_value"),
        _min("full_date").alias("first_txn_date"),
        _max("full_date").alias("last_txn_date"),
        countDistinct("stock_id").alias("unique_stocks_traded"),
        countDistinct("full_date").alias("active_trading_days")
    )
)

# Holding duration: avg days between BUY and next SELL per user per stock
df_buys = (
    df_txn_dated.filter(col("txn_type") == "BUY")
    .select("user_id", "stock_id", col("full_date").alias("buy_date"))
    .withColumn("buy_rank", row_number().over(
        Window.partitionBy("user_id", "stock_id").orderBy("buy_date")))
)

df_sells = (
    df_txn_dated.filter(col("txn_type") == "SELL")
    .select("user_id", "stock_id", col("full_date").alias("sell_date"))
    .withColumn("sell_rank", row_number().over(
        Window.partitionBy("user_id", "stock_id").orderBy("sell_date")))
)

df_holding = (
    df_buys.join(df_sells,
                 (df_buys.user_id == df_sells.user_id) &
                 (df_buys.stock_id == df_sells.stock_id) &
                 (df_buys.buy_rank == df_sells.sell_rank), "inner")
    .withColumn("holding_days", datediff("sell_date", "buy_date"))
    .groupBy(df_buys.user_id)
    .agg(
        _round(avg("holding_days"), 1).alias("avg_holding_days"),
        _min("holding_days").alias("min_holding_days"),
        _max("holding_days").alias("max_holding_days"),
        count("*").alias("completed_trades")
    )
)

# Combine and classify
df_patterns = (
    df_user_stats
    .join(df_holding, "user_id", "left")
    .fillna(0)
    .withColumn("investment_span_days", datediff("last_txn_date", "first_txn_date"))
    .withColumn("txn_frequency", _round(
        when(col("investment_span_days") > 0,
             col("total_txns") / col("investment_span_days") * 30  # txns per month
        ).otherwise(col("total_txns")), 2))
    .withColumn("investor_type",
        when((col("txn_frequency") >= 10) | (col("avg_holding_days") < 7), "High-Frequency Trader")
        .when((col("avg_holding_days") >= 30) | (col("txn_frequency") < 3), "Long-Term Investor")
        .otherwise("Moderate Trader")
    )
    .select(
        "user_id", "total_txns", "buy_txns", "sell_txns",
        "total_value_traded", "avg_txn_value",
        "first_txn_date", "last_txn_date", "investment_span_days",
        "unique_stocks_traded", "active_trading_days",
        "avg_holding_days", "min_holding_days", "max_holding_days",
        "completed_trades", "txn_frequency", "investor_type"
    )
)

print("6\ufe0f\u20e3 USER INVESTMENT PATTERNS")
upsert(df_patterns, "user_investment_patterns", "user_id")

In [0]:
# =========================
# SUMMARY — ALL GOLD TABLES
# =========================

gold_tables = [
    "stock_profitability",
    "portfolio_performance",
    "trading_behavior",
    "market_event_impact",
    "sector_performance",
    "user_investment_patterns"
]

print("=" * 60)
print(" GOLD LAYER SUMMARY")
print("=" * 60)
for tbl in gold_tables:
    full = f"{GOLD}.{tbl}"
    if spark.catalog.tableExists(full):
        cnt = spark.table(full).count()
        cols = len(spark.table(full).columns)
        print(f"   {full:<45} {cnt:>8,} rows  |  {cols} cols")
    else:
        print(f"   {full:<45} not yet created")
print("=" * 60)
print(" Gold layer build complete!")